In [3]:
import pandas as pd
import tkinter as tk
from tkinter import filedialog
import os
import sys

def procesar_excel():
    # --- 1. CONFIGURACIÓN DE VENTANA (Siempre visible) ---
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True) 
    
    print("Abriendo ventana de selección...")
    root.update()
    
    file_path = filedialog.askopenfilename(
        title="Seleccionar archivo Excel (Transacciones)",
        filetypes=[("Archivos de Excel", "*.xlsx *.xls")]
    )
    root.destroy()

    if not file_path:
        print("No se seleccionó ningún archivo.")
        return

    try:
        print(f"Procesando: {os.path.basename(file_path)}...")
        
        # --- 2. CARGA DE DATOS ---
        df = pd.read_excel(file_path, sheet_name="Transacciones")
        
        # --- 3. RENOMBRAR COLUMNA FECHA ---
        if 'Fecha' in df.columns:
            df.rename(columns={'Fecha': 'Fecha y hora'}, inplace=True)

        # --- 4. TRANSFORMACIÓN DE TIPOS DE DATOS (PANDAS) ---
        
        # A) Nro. Transacción -> TEXTO (String)
        # Importante: Lo convertimos a string aquí para que Pandas no lo trate como número
        if 'Nro. Transacción' in df.columns:
            df['Nro. Transacción'] = df['Nro. Transacción'].astype(str).str.replace(r'\.0$', '', regex=True)

        # B) Fechas -> DATETIME REAL
        df['Fecha y hora'] = pd.to_datetime(df['Fecha y hora'], dayfirst=True, errors='coerce')
        # Creamos la columna Fecha (solo fecha, sin hora, pero sigue siendo objeto fecha)
        df['Fecha'] = df['Fecha y hora'].dt.normalize()

        # C) Monto total -> NÚMERO (Float/Int)
        if 'Monto total' in df.columns:
            df['Monto total'] = pd.to_numeric(df['Monto total'], errors='coerce').fillna(0)

        # --- 5. LIMPIEZA DE COLUMNAS ---
        columnas_a_quitar = [
            "Ticket original", "Fecha original", "Nombre de aplicación EMV", 
            "Id aplicación EMV", "Tipo de cuenta", "Tarjetahabiente", 
            "Origen de tarjeta", "Pregunta inicial", "Grupo", "Dirección", 
            "Locacion", "Departamento", "Mayorista", "Informacion de venta", 
            "Id usuario", "Id QR", "Transacciones relacionadas", 
            "Meses diferidos", "Cuotas", "Factura", "Branch"
        ]
        df.drop(columns=[col for col in columnas_a_quitar if col in df.columns], inplace=True)

        # --- 6. REORDENAR ---
        cols = list(df.columns)
        if 'Fecha' in cols and 'Fecha y hora' in cols:
            cols.remove('Fecha')
            idx = cols.index('Fecha y hora') + 1
            cols.insert(idx, 'Fecha')
            df = df[cols]

        # --- 7. GUARDAR CON FORMATOS ESPECÍFICOS (XLSXWRITER) ---
        root_save = tk.Tk()
        root_save.withdraw()
        root_save.attributes('-topmost', True)
        root_save.update()

        print("Selecciona dónde guardar...")
        save_path = filedialog.asksaveasfilename(
            title="Guardar archivo limpio",
            defaultextension=".xlsx",
            filetypes=[("Archivos de Excel", "*.xlsx")],
            initialfile="Transacciones_Procesado.xlsx"
        )
        root_save.destroy()

        if save_path:
            # Usamos xlsxwriter para tener control total de los formatos
            with pd.ExcelWriter(save_path, engine='xlsxwriter') as writer:
                df.to_excel(writer, index=False, sheet_name='Transacciones')
                
                workbook = writer.book
                worksheet = writer.sheets['Transacciones']
                
                # --- DEFINICIÓN DE FORMATOS ---
                formato_texto = workbook.add_format({'num_format': '@'}) # El @ fuerza texto en Excel
                formato_fecha = workbook.add_format({'num_format': 'dd/mm/yyyy'})
                formato_fecha_hora = workbook.add_format({'num_format': 'dd/mm/yyyy hh:mm:ss'})
                formato_numero = workbook.add_format({'num_format': '#,##0'}) # Sin decimales o usar '#,##0.00'
                
                # --- APLICACIÓN DE FORMATOS POR COLUMNA ---
                for idx, col_name in enumerate(df.columns):
                    # Calculamos el ancho ideal (estética)
                    max_len = max(df[col_name].astype(str).map(len).max(), len(col_name)) + 2
                    
                    # Aplicamos formato según el nombre de la columna
                    if col_name == 'Nro. Transacción':
                        worksheet.set_column(idx, idx, max_len, formato_texto)
                    
                    elif col_name == 'Fecha':
                        worksheet.set_column(idx, idx, max_len, formato_fecha)
                        
                    elif col_name == 'Fecha y hora':
                        worksheet.set_column(idx, idx, max_len + 5, formato_fecha_hora)
                        
                    elif col_name == 'Monto total':
                        worksheet.set_column(idx, idx, max_len, formato_numero)
                    
                    else:
                        # Resto de columnas (formato general)
                        worksheet.set_column(idx, idx, max_len)

            print(f"¡Éxito! Archivo guardado correctamente en: {save_path}")

    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
        input("Presiona Enter para cerrar...")

if __name__ == "__main__":
    procesar_excel()

Abriendo ventana de selección...
Procesando: Transacciones_202602101047221.xlsx...
Selecciona dónde guardar...
¡Éxito! Archivo guardado correctamente en: G:/Mi unidad/Modelo de automatizaciones/Las Invernadas Paula/Transacciones_Procesado.xlsx
